In [ ]:
import joblib
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import MACCSkeys
from rdkit import DataStructs
from mordred import Calculator, descriptors
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
import os
import json
import os
import warnings
from mordred import Calculator, descriptors as mordred_descriptors
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

INPUT_CSV = "Analogues_PF2.csv"          # CSV de compuestos nuevos (smiles_std, compound_id)
OUTPUT_CSV = "predicciones_analogos"
os.makedirs(OUTPUT_CSV, exist_ok=True)

In [3]:
def canonicalize_smiles(df: pd.DataFrame, smiles_col: str = "smiles_std") -> pd.DataFrame:
    df = df.copy()
    # limpieza de espacios/saltos de linea que a veces se cuelan al copiar SMILES
    df[smiles_col] = df[smiles_col].astype(str).str.replace(r"\s+", "", regex=True)

    mols, canon_smiles, valid_idx = [], [], []
    for i, smi in enumerate(df[smiles_col]):
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            mols.append(mol)
            canon_smiles.append(Chem.MolToSmiles(mol))
            valid_idx.append(i)

    n_invalid = len(df) - len(valid_idx)
    if n_invalid > 0:
        print(f"[AVISO] {n_invalid} SMILES invalidos descartados de {len(df)}.")

    df_valid = df.iloc[valid_idx].reset_index(drop=True)
    df_valid["smiles_canonical"] = canon_smiles
    return df_valid, mols


df_new = pd.read_csv(INPUT_CSV)
df_new, mols = canonicalize_smiles(df_new, smiles_col="smiles_std")
print(f"Compuestos validos: {len(df_new)}")
df_new.head()


Compuestos validos: 39


,smiles_std,compound_id,smiles_canonical
0,CN(C)[C@@H]([C@H](CC)C)C(O[C@@H](CC(C)C)C(N[C@...,gallinamide_analogo_novo_1,CC[C@H](C)[C@@H](C(=O)O[C@@H](CC(C)C)C(=O)N[C@...
1,CN(C)[C@@H](C(C)C)C(O[C@@H](CC(C)C)C(N[C@@H](C...,gallinamide_analogo_novo_2,COC1=CC(=O)N(C(=O)/C=C/[C@H](C)NC(=O)[C@H](CC(...
2,CN(C)[C@@H]([C@H](CC)C)C(O[C@@H](CC(C)C)C(N[C@...,gallinamide_analogo_novo_ 3,CC[C@H](C)[C@@H](C(=O)O[C@@H](CC(C)C)C(=O)N[C@...
3,CN(C)[C@@H](CC1=CC=CC=C1)C(O[C@@H](CC(C)C)C(N[...,gallinamide_analogo_n ovo_4,COC1=CC(=O)N(C(=O)/C=C/[C@H](C)NC(=O)[C@H](CC(...
4,CN(C)[C@@H]([C@@H](C)CC)C(O[C@@H](CC(C)C)C(N[C...,gallinamide_analogo_novo_5,CC[C@H](C)[C@@H](C(=O)O[C@@H](CC(C)C)C(=O)N[C@...


In [11]:
maccs_bundle = joblib.load("maccs_bundle.joblib")
mordred_bundle = joblib.load("mordred_bundle.joblib")

for name, bundle in [("MACCS", maccs_bundle), ("Mordred", mordred_bundle)]:
    print(f"--- {name} ---")
    print(f"  n_features      : {len(bundle['feature_columns'])}")
    print(f"  descriptor_type : {bundle['descriptor_type']}")
    print(f"  modelos top-3   : {list(bundle['top3_models'].keys())}")
    print(f"  AD threshold    : {bundle['ad_threshold']:.4f}  (k={bundle['ad_n_neighbors']})")
    print(f"  label_meaning   : {bundle['label_meaning']}")
    print()

--- MACCS ---
  n_features      : 167
  descriptor_type : maccs
  modelos top-3   : ['SVM_RBF', 'AdaBoost', 'SVM_Linear']
  AD threshold    : 13.5531  (k=5)
  label_meaning   : {'1': 'Activo (IC50 < 5 uM frente a Falcipaina-2)', '0': 'Inactivo (IC50 >= 5 uM frente a Falcipaina-2)'}

--- Mordred ---
  n_features      : 317
  descriptor_type : mordred
  modelos top-3   : ['KNeighbors', 'GradientBoosting', 'ExtraTrees']
  AD threshold    : 14.9951  (k=5)
  label_meaning   : {'1': 'Activo (IC50 < 5 uM frente a Falcipaina-2)', '0': 'Inactivo (IC50 >= 5 uM frente a Falcipaina-2)'}



In [13]:
from rdkit.Chem import MACCSkeys
from rdkit import DataStructs

def compute_maccs_df(smiles_list):
    rows, valid_idx = [], []
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        fp = MACCSkeys.GenMACCSKeys(mol)
        arr = np.zeros((167,), dtype=int)
        DataStructs.ConvertToNumpyArray(fp, arr)
        rows.append(arr)
        valid_idx.append(i)
    cols = [f"MACCS_{i}" for i in range(167)]
    X = pd.DataFrame(rows, columns=cols)
    return X, valid_idx

print("Calculando MACCS keys para los candidatos...")
X_maccs_raw, valid_idx_maccs = compute_maccs_df(df_new["smiles_std"].tolist())
df_maccs_cand = df_new.iloc[valid_idx_maccs].reset_index(drop=True)
X_maccs = X_maccs_raw.reindex(columns=maccs_bundle["feature_columns"])

mask_completo = X_maccs.notna().all(axis=1)
n_incompletos = int((~mask_completo).sum())
if n_incompletos:
    print(f"  - descartadas por features MACCS incompletas (sin imputar): {n_incompletos}")

X_maccs = X_maccs[mask_completo].reset_index(drop=True)
df_maccs_cand = df_maccs_cand[mask_completo].reset_index(drop=True)

print(f"Matriz MACCS candidatos: {X_maccs.shape}")

Calculando MACCS keys para los candidatos...
Matriz MACCS candidatos: (39, 167)


In [14]:
from mordred import Calculator, descriptors

mols_mordred, valid_idx_mordred = [], []
for i, smi in enumerate(df_new["smiles_std"]):
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        mols_mordred.append(mol)
        valid_idx_mordred.append(i)

df_mordred_cand = df_new.iloc[valid_idx_mordred].reset_index(drop=True)

print(f"Calculando descriptores Mordred 2D para {len(mols_mordred)} moléculas...")
calc = Calculator(descriptors, ignore_3D=True)
df_desc_raw = calc.pandas(mols_mordred)

X_mordred = df_desc_raw.apply(pd.to_numeric, errors="coerce")
X_mordred = X_mordred.reindex(columns=mordred_bundle["feature_columns"])

mask_completo = X_mordred.notna().all(axis=1)
n_incompletos = int((~mask_completo).sum())
if n_incompletos:
    print(f"  - descartadas por descriptores Mordred incompletos (sin imputar): {n_incompletos}")

X_mordred = X_mordred[mask_completo].reset_index(drop=True)
df_mordred_cand = df_mordred_cand[mask_completo].reset_index(drop=True)

print(f"Matriz Mordred candidatos: {X_mordred.shape}")


Calculando descriptores Mordred 2D para 39 moléculas...


 28%|██▊       | 11/39 [00:02<00:04,  6.81it/s]

c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 56%|█████▋    | 22/39 [00:02<00:01, 13.92it/s]

c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


...

c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 92%|█████████▏| 36/39 [00:03<00:00, 16.48it/s]

c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


100%|██████████| 39/39 [00:03<00:00,  9.90it/s]


c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
Matriz Mordred candidatos: (39, 317)


In [15]:
def apply_ad(X, bundle):
    X_scaled = bundle["ad_scaler"].transform(X)
    dist, _ = bundle["ad_nn"].kneighbors(X_scaled, n_neighbors=bundle["ad_n_neighbors"])
    mean_dist = dist.mean(axis=1)
    in_ad = mean_dist < bundle["ad_threshold"]
    return mean_dist, in_ad

dist_maccs, in_ad_maccs = apply_ad(X_maccs, maccs_bundle)
dist_mordred, in_ad_mordred = apply_ad(X_mordred, mordred_bundle)

df_maccs_cand["ad_dist"] = dist_maccs
df_maccs_cand["in_AD_maccs"] = in_ad_maccs

df_mordred_cand["ad_dist"] = dist_mordred
df_mordred_cand["in_AD_mordred"] = in_ad_mordred

print(f"MACCS   - dentro del AD: {in_ad_maccs.sum()} / {len(in_ad_maccs)} ({in_ad_maccs.mean():.1%})")
print(f"Mordred - dentro del AD: {in_ad_mordred.sum()} / {len(in_ad_mordred)} ({in_ad_mordred.mean():.1%})")

MACCS   - dentro del AD: 39 / 39 (100.0%)
Mordred - dentro del AD: 39 / 39 (100.0%)


In [19]:
def predict_bundle(X, bundle):
    """Devuelve prob/pred del top-1 y el nº de votos 'activo' entre los top-3."""
    ranking = bundle["top3_ranking"]     
    models  = bundle["top3_models"]

    top1_name = ranking[0]["model"]
    top1_model = models[top1_name]
    if hasattr(top1_model, "predict_proba"):
        prob_top1 = top1_model.predict_proba(X)[:, 1]
    else:
        prob_top1 = top1_model.decision_function(X)
    pred_top1 = (prob_top1 >= 0.5).astype(int)

    votes = np.zeros(len(X), dtype=int)
    for rank_info in ranking:
        m = models[rank_info["model"]]
        if hasattr(m, "predict_proba"):
            p = m.predict_proba(X)[:, 1]
        else:
            p = m.decision_function(X)
        votes += (p >= 0.5).astype(int)

    return prob_top1, pred_top1, votes, top1_name

prob_maccs, pred_maccs, votes_maccs, name_maccs = predict_bundle(X_maccs, maccs_bundle)
prob_mordred, pred_mordred, votes_mordred, name_mordred = predict_bundle(X_mordred, mordred_bundle)

df_maccs_cand["maccs_model_top1"]   = name_maccs
df_maccs_cand["maccs_prob_top1"]    = prob_maccs
df_maccs_cand["maccs_pred_top1"]    = pred_maccs
df_maccs_cand["maccs_top3_votes"]   = votes_maccs   

df_mordred_cand["mordred_model_top1"] = name_mordred
df_mordred_cand["mordred_prob_top1"]  = prob_mordred
df_mordred_cand["mordred_pred_top1"]  = pred_mordred
df_mordred_cand["mordred_top3_votes"] = votes_mordred

print(f"MACCS   top-1 = {name_maccs}   | predichos activos: {pred_maccs.sum()} / {len(pred_maccs)}")
print(f"Mordred top-1 = {name_mordred} | predichos activos: {pred_mordred.sum()} / {len(pred_mordred)}")

# --- Combinar ambos featurizadores por molécula (join por smiles_std) --------
cols_maccs = ["compound_id" , "smiles_std", "in_AD_maccs", "ad_dist",
              "maccs_model_top1", "maccs_prob_top1", "maccs_pred_top1", "maccs_top3_votes"]
cols_mordred = ["compound_id" , "smiles_std", "in_AD_mordred", "ad_dist",
                "mordred_model_top1", "mordred_prob_top1", "mordred_pred_top1", "mordred_top3_votes"]

df_m1 = df_maccs_cand[cols_maccs].rename(columns={"ad_dist": "ad_dist_maccs"})
df_m2 = df_mordred_cand[cols_mordred].rename(columns={"ad_dist": "ad_dist_mordred"})

df_final = df_m1.merge(df_m2, on=["compound_id" , "smiles_std"], how="inner")
print(f"Moléculas con MACCS y Mordred calculados con éxito: {len(df_final)}")

# --- Consenso estricto --------------------------------------------------------
df_final["in_AD_ambos"] = df_final["in_AD_maccs"] & df_final["in_AD_mordred"]
df_final["activo_ambos_top1"] = (df_final["maccs_pred_top1"] == 1) & (df_final["mordred_pred_top1"] == 1)
df_final["candidato_MoA_FP2"] = df_final["in_AD_ambos"] & df_final["activo_ambos_top1"]

df_final["prob_media"] = (df_final["maccs_prob_top1"] + df_final["mordred_prob_top1"]) / 2

df_final = df_final.sort_values("prob_media", ascending=False).reset_index(drop=True)


df_final.to_csv(f'{OUTPUT_CSV}/df_descriptores.csv', index=False)

df_final.head(20)


MACCS   top-1 = SVM_RBF   | predichos activos: 39 / 39
Mordred top-1 = KNeighbors | predichos activos: 39 / 39
Moléculas con MACCS y Mordred calculados con éxito: 39


,compound_id,smiles_std,in_AD_maccs,ad_dist_maccs,maccs_model_top1,maccs_prob_top1,maccs_pred_top1,maccs_top3_votes,in_AD_mordred,ad_dist_mordred,mordred_model_top1,mordred_prob_top1,mordred_pred_top1,mordred_top3_votes,in_AD_ambos,activo_ambos_top1,candidato_MoA_FP2,prob_media
0,gallinami de_analogo_novo_36,O=C1N(C(/C=C/[C@H](CCC2=CC=CC=C2)NC([C@H](CC(C...,True,6.689078,SVM_RBF,0.929904,1,3,True,9.761731,KNeighbors,1.0,1,3,True,True,True,0.964952
1,gallinamid e_analogo_novo_34,O=C1N(C(/C=C/[C@H](CCC2=CC=CC=C2)NC([C@H](CC(C...,True,5.986121,SVM_RBF,0.917682,1,3,True,8.709319,KNeighbors,1.0,1,3,True,True,True,0.958841
2,gallinamide_an alogo_novo_30,O=C1N(C(/C=C/[C@H](CCC2=CC=CC=C2)NC([C@H](CC(C...,True,5.764340,SVM_RBF,0.914816,1,3,True,8.496380,KNeighbors,1.0,1,3,True,True,True,0.957408
3,g allinamide_analogo_novo_38,O=C1N(C(/C=C/[C@H](CCC2=CC=CC=C2)NC([C@H](CC(C...,True,5.090487,SVM_RBF,0.900427,1,3,True,9.751441,KNeighbors,1.0,1,3,True,True,True,0.950214
4,gallinamide_analogo_novo_20,CN([C@H](C(O[C@H](C(N[C@H](C(N[C@H](/C=C/C(N1C...,True,5.235991,SVM_RBF,0.896267,1,3,True,4.629129,KNeighbors,1.0,1,3,True,True,True,0.948134
5,gallinamide_analogo_novo_2,CN(C)[C@@H](C(C)C)C(O[C@@H](CC(C)C)C(N[C@@H](C...,True,5.235991,SVM_RBF,0.896267,1,3,True,4.629129,KNeighbors,1.0,1,3,True,True,True,0.948134
6,gallinamide_analogo _novo_28,C[C@@H](/C=C/C(N1[C@@H](CC2=CNC3=C2C=CC=C3)C(O...,True,2.646763,SVM_RBF,0.871678,1,3,True,3.575035,KNeighbors,1.0,1,3,True,True,True,0.935839
7,gallinamide _analogo_novo_8,CN(C)[C@@H]([C@H](CC)C)C(O[C@@H](CC(C)C)C(N[C@...,True,7.326978,SVM_RBF,0.857128,1,3,True,7.910837,KNeighbors,1.0,1,3,True,True,True,0.928564
8,gallinamide_analogo_ novo_26,CN([C@H](C(O[C@H](C(N[C@H](C(N[C@@H](/C=C/C(N1...,True,7.326978,SVM_RBF,0.857128,1,3,True,7.910837,KNeighbors,1.0,1,3,True,True,True,0.928564
9,gallinamide_analog o_novo_25,CN([C@H](C(O[C@H](C(N[C@H](C(N[C@@H](/C=C/C(N1...,True,7.326978,SVM_RBF,0.857128,1,3,True,7.910837,KNeighbors,1.0,1,3,True,True,True,0.928564
